# 03 — Classical Feature Extraction

Extract ~194 classical ML features from preprocessed 1 s segments (16 kHz, mono). Outputs a row per segment with label and feature columns, saved to CSV/Parquet for classical model training.

In [1]:
# Imports and setup
from pathlib import Path
import yaml, json
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
CFG_PATH = PROJECT_ROOT / 'config.yaml'
PROC_DIR = PROJECT_ROOT / 'data' / 'processed' / 'universal'
OUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'classical_features'
OUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR = PROJECT_ROOT / 'results' / 'metrics'
METRICS_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)


Project root: /Users/harryirving/Development/projects/ai-ml/BikeAIv4


In [2]:
# Load config (supports  or config:)
import yaml
with open(CFG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)
data_cfg = (cfg.get('data') or cfg.get('config', {}).get('data') or cfg.get('config') or {})
audio_cfg = (cfg.get('audio') or cfg.get('config', {}).get('audio') or {})
TARGET_SR = int(audio_cfg.get('sample_rate', 16000))
CLASSES = ['angle_grinder', 'background', 'tools']
print('Processed universal dir:', PROC_DIR)


Processed universal dir: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/data/processed/universal


## Feature definitions

In [3]:
def feature_vector(y, sr):
    # Safety
    if y is None or len(y) == 0:
        return None
    # Basic spectral params
    n_fft = 2048
    hop = 512
    # Time-domain features
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=n_fft, hop_length=hop)[0]
    rms = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop, center=True)[0]
    # Spectral features
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop))
    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(S=S, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(S=S, sr=sr, roll_percent=0.85)[0]
    contrast = librosa.feature.spectral_contrast(S=S, sr=sr, fmin=50, n_bands=6)
    flatness = librosa.feature.spectral_flatness(S=S)[0]
    # MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=n_fft, hop_length=hop)
    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_dd = librosa.feature.delta(mfcc, order=2)

    def stats(x):
        return {
            'mean': float(np.mean(x)),
            'std': float(np.std(x)),
            'min': float(np.min(x)),
            'max': float(np.max(x)),
            'median': float(np.median(x)),
            'p10': float(np.percentile(x, 10)),
            'p90': float(np.percentile(x, 90)),
        }
    feats = {}
    # Aggregate stats for each feature stream
    for name, arr in [
        ('zcr', zcr), ('rms', rms), ('centroid', centroid), ('bandwidth', bandwidth), ('rolloff', rolloff), ('flatness', flatness)
    ]:
        s = stats(arr)
        for k, v in s.items():
            feats[f'{name}_{k}'] = v
    # Spectral contrast: 7 bands x stats
    for i in range(contrast.shape[0]):
        s = stats(contrast[i])
        for k, v in s.items():
            feats[f'contrast_b{i}_{k}'] = v
    # MFCCs: 13 coeffs x stats
    for i in range(mfcc.shape[0]):
        s = stats(mfcc[i])
        for k, v in s.items():
            feats[f'mfcc{i+1}_{k}'] = v
    # Delta and Delta-Delta
    for i in range(mfcc_delta.shape[0]):
        s = stats(mfcc_delta[i])
        for k, v in s.items():
            feats[f'mfcc_delta{i+1}_{k}'] = v
    for i in range(mfcc_dd.shape[0]):
        s = stats(mfcc_dd[i])
        for k, v in s.items():
            feats[f'mfcc_dd{i+1}_{k}'] = v
    return feats


## Walk preprocessed dataset and extract features

In [4]:
AUDIO_EXTS = ('.wav',)
CLASSES = ['angle_grinder', 'background', 'tools']
rows = []
for label in CLASSES:
    cls_dir = PROC_DIR / label
    if not cls_dir.exists():
        print('Skipping missing class dir:', cls_dir)
        continue
    files = sorted([p for p in cls_dir.rglob('*') if p.suffix.lower() in AUDIO_EXTS])
    print(f'Extracting {label}: {len(files)} files')
    for p in tqdm(files):
        try:
            y, sr = sf.read(p)
            if y.ndim > 1:
                y = librosa.to_mono(y.T)
            feats = feature_vector(y.astype(np.float32), sr)
            if feats is None:
                continue
            feats['label'] = label
            feats['path'] = str(p)
            rows.append(feats)
        except Exception as e:
            print('Error:', p, e)
            continue
df_feats = pd.DataFrame(rows)
print('Total rows:', len(df_feats))
df_feats.head()


Extracting angle_grinder: 6378 files


  0%|          | 0/6378 [00:00<?, ?it/s]

Extracting background: 8069 files


  0%|          | 0/8069 [00:00<?, ?it/s]

Extracting tools: 4597 files


  0%|          | 0/4597 [00:00<?, ?it/s]

Total rows: 19044


,zcr_mean,zcr_std,zcr_min,zcr_max,zcr_median,zcr_p10,zcr_p90,rms_mean,rms_std,rms_min,...,mfcc_dd12_p90,mfcc_dd13_mean,mfcc_dd13_std,mfcc_dd13_min,mfcc_dd13_max,mfcc_dd13_median,mfcc_dd13_p10,mfcc_dd13_p90,label,path
0,0.318863,0.041395,0.173340,0.365234,0.327393,0.286426,0.354346,0.169749,0.050727,0.052280,...,0.314156,0.195233,0.574609,-0.906352,1.282232,0.157533,-0.679947,0.806133,angle_grinder,/Users/harryirving/Development/projects/ai-ml/...
1,0.319702,0.047375,0.149414,0.364258,0.331543,0.267383,0.357910,0.047645,0.004539,0.038186,...,0.657159,0.077244,0.320415,-0.457496,1.059320,0.052150,-0.358967,0.465961,angle_grinder,/Users/harryirving/Development/projects/ai-ml/...
2,0.310852,0.043940,0.166504,0.363770,0.321777,0.262744,0.351709,0.043870,0.005593,0.030816,...,0.491295,0.226140,0.459097,-0.765372,1.013577,0.342826,-0.441053,0.686009,angle_grinder,/Users/harryirving/Development/projects/ai-ml/...
3,0.297592,0.038305,0.163574,0.328125,0.310059,0.255859,0.322754,0.043762,0.004898,0.031284,...,0.833643,0.206188,0.535757,-0.764109,0.924751,0.209616,-0.619733,0.924751,angle_grinder,/Users/harryirving/Development/projects/ai-ml/...
4,0.291794,0.039925,0.160645,0.327637,0.305908,0.244189,0.316309,0.042453,0.005029,0.027438,...,0.775515,0.017446,0.645951,-1.069406,1.030754,0.146723,-0.612807,0.787783,angle_grinder,/Users/harryirving/Development/projects/ai-ml/...


## Save features

In [5]:
csv_path = OUT_DIR / 'classical_features.csv'
parquet_path = OUT_DIR / 'classical_features.parquet'
df_feats.to_csv(csv_path, index=False)
try:
    df_feats.to_parquet(parquet_path, index=False)
except Exception as e:
    print('Parquet save skipped:', e)
print('Saved:', csv_path)


Saved: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/data/processed/classical_features/classical_features.csv
